# NanoGPT — Complete Reasoning for Every Decision

This document explains **why** every single choice was made —  
architecture, hyperparameters, and every line of PyTorch code.

---

## How to use this
Read this alongside the implementation notebook.  
Every section maps to a step in the code.  
By the end you should be able to answer any interview question about transformers.


---
# PART 1 — DATA AND TOKENISER
---

## Why character-level tokenisation?

There are three main ways to tokenise text:

| Method | Example: 'playing' | Vocab size | Pros | Cons |
|--------|-------------------|------------|------|------|
| **Character-level** | ['p','l','a','y','i','n','g'] | ~65–200 | Simple, no OOV | Long sequences |
| **Word-level** | ['playing'] | 50K–200K | Short sequences | Huge vocab, OOV problem |
| **BPE (subword)** | ['play','ing'] | 32K–100K | Balance of both | Complex to implement |

We use **character-level** because:
1. **Tiny Shakespeare is small** (~1M chars) — word-level would have too many rare words
2. **Simple to implement** — the entire tokeniser is two dicts and two functions
3. **Educational** — forces the model to learn spelling, grammar, punctuation from scratch
4. **No out-of-vocabulary problem** — every possible character is in the vocab

GPT-2/3/4 use **BPE** in production because it's more efficient — 'playing' as 2 tokens instead of 7 means shorter sequences and faster training. But for a learning project, character-level is perfect.

## Why sort the vocabulary?

```python
chars = sorted(list(set(text)))
```

`set(text)` removes duplicates — we only want each character once.  
`sorted()` ensures the mapping is **deterministic** — if you re-run the code, 'a' always maps to the same integer. Without sorting, Python sets have random ordering and your saved model would be useless after a restart.

## Why encode the entire dataset into a tensor upfront?

```python
data = torch.tensor(encode(text), dtype=torch.long)
```

Alternative: encode each batch on-the-fly during training.  
Problem: string processing in Python is slow. Doing it millions of times during training adds significant overhead.  
Solution: encode once, store as a tensor. Batch sampling then becomes a fast tensor slice operation.

`dtype=torch.long` = int64. We use int64 (not int32) because `nn.Embedding` in PyTorch requires long integer indices.

## Why 90/10 train/val split?

```python
n = int(0.9 * len(data))
```

Tiny Shakespeare is only ~1M characters. A 10% val set gives ~100K characters — enough to get a stable loss estimate. Using more val data would waste training signal; using less would give noisy val loss estimates.  

**Critical:** we take the LAST 10% as validation, not a random sample. This simulates real deployment — the model is trained on earlier text and tested on later text. Random sampling would let information leak between train and val (the same scene might appear in both).

---
# PART 2 — BATCH SAMPLING
---

## Why random starting positions?

```python
ix = torch.randint(len(data_src) - block_size, (batch_size,))
```

We subtract `block_size` from the upper bound because each sequence needs `block_size` tokens AFTER the start position. If data has 1000 tokens and block_size=256, the last valid start is position 744 (744+256=1000).

We sample randomly (not sequentially) because:
- Sequential batching would mean the model sees the entire dataset in order every epoch
- Adjacent batches would be correlated — violates the i.i.d. assumption of SGD
- Random sampling ensures each batch is an independent sample from the data distribution

## Why does one sequence give multiple training examples?

```python
x = torch.stack([data_src[i:i+block_size]   for i in ix])
y = torch.stack([data_src[i+1:i+block_size+1] for i in ix])
```

For a sequence `[t0, t1, t2, t3]` with block_size=4:
```
x = [t0, t1, t2, t3]
y = [t1, t2, t3, t4]
```

This creates 4 training examples from one sequence:
```
context=[t0]         → predict t1
context=[t0,t1]      → predict t2  
context=[t0,t1,t2]   → predict t3
context=[t0,t1,t2,t3]→ predict t4
```

This is extremely data-efficient. One sequence of length T gives T training examples.  
It also means the model is trained to predict the next token regardless of context length — it works with 1 token of context or 256.

## Why @torch.no_grad() in estimate_loss?

```python
@torch.no_grad()
def estimate_loss(model, ...):
```

During evaluation we only need the forward pass — no gradients needed.  
PyTorch by default tracks all operations to build a computation graph for backprop.  
`no_grad()` disables this, which:
- Reduces memory usage by ~50% (no gradient tensors stored)
- Speeds up computation by ~30%
- Prevents accidental gradient accumulation during eval

## Why model.eval() before evaluation?

```python
model.eval()
```

Two layers behave differently in train vs eval mode:
1. **Dropout**: In train mode, randomly zeros activations. In eval mode, passes all activations through (no randomness). During evaluation we want deterministic, repeatable results.
2. **BatchNorm**: Uses running statistics in eval, batch statistics in train. (We don't use BatchNorm, but this is the general reason.)

Without `model.eval()`, your validation loss would be slightly different every time you run it — even with the same data — because dropout introduces randomness.

---
# PART 3 — BIGRAM MODEL
---

## Why build a bigram model at all?

Two reasons:
1. **Baseline**: We need a lower bound. If our GPT doesn't significantly beat bigram, something is wrong.
2. **Shared interface**: The bigram model uses the exact same `forward(idx, targets)` and `generate()` interface as the full GPT. This lets us verify the training loop, loss function, and generation code on a simple model before adding complexity.

## Why is nn.Embedding(vocab_size, vocab_size) the bigram model?

```python
self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
```

`nn.Embedding(A, B)` is a lookup table of shape (A, B).  
When you index it with integer `i`, you get row `i` — a vector of length B.  

Here A=B=vocab_size=65. So:  
- Row 0 = logits for "what comes after character 0"
- Row 1 = logits for "what comes after character 1"
- ...

This is literally a bigram count table, but learned via gradient descent instead of counting.

## Why reshape for cross_entropy?

```python
B, T, C = logits.shape
logits  = logits.view(B*T, C)
targets = targets.view(B*T)
```

`F.cross_entropy` expects inputs of shape `(N, C)` where N = number of examples, C = number of classes.  
Our logits are `(B, T, C)` — batch × time × vocab.  
We flatten B and T into one dimension: `(B*T, C)` — each token position becomes one example.  
Targets similarly flatten from `(B, T)` to `(B*T,)`.  

This is equivalent to treating each token prediction as an independent classification problem.

## What does cross-entropy loss actually compute?

```python
loss = F.cross_entropy(logits, targets)
```

For each position: `loss = -log(softmax(logits)[correct_token])`

Intuition:
- If model assigns probability 1.0 to correct token → loss = -log(1) = 0 (perfect)
- If model assigns probability 0.5 → loss = -log(0.5) = 0.69
- If model assigns probability 0.01 → loss = -log(0.01) = 4.6 (terrible)

For a random model with 65 characters: each token has prob 1/65 ≈ 0.015  
Expected loss = -log(1/65) = log(65) ≈ 4.17  
This is why we expect loss ~4.17 at initialisation.

## Why torch.multinomial instead of argmax for generation?

```python
idx_next = torch.multinomial(probs, num_samples=1)
```

**argmax** (greedy): always pick the highest probability token.  
Problem: the model quickly falls into repetitive loops. If 'the' always follows 'in', it generates 'in the in the in the...'  

**multinomial** (sampling): sample from the probability distribution.  
If 'e' has prob 0.6 and 'a' has prob 0.3, sometimes we pick 'a'.  
This creates diverse, natural-sounding text.  

Temperature later controls how sharp or flat this distribution is.

---
# PART 4 — SELF-ATTENTION
---

## Why do we need attention at all?

In a bigram model, each token only sees itself.  
In an RNN, tokens can see the past but information decays over distance.  
In attention, every token can directly attend to every other past token — no decay.

The fundamental problem attention solves:  
*"ROMEO: I love ___"* — to predict the next word, we need to know who's speaking (ROMEO, 20 tokens ago). Attention can connect any two positions directly.

## Why Q, K, V? Why three separate projections?

```python
self.key   = nn.Linear(n_embd, head_size, bias=False)
self.query = nn.Linear(n_embd, head_size, bias=False)
self.value = nn.Linear(n_embd, head_size, bias=False)
```

Think of it like a library:
- **Query**: Your search term — "I'm looking for information about the subject of this sentence"
- **Key**: The index card — "This token contains information about a noun/subject"
- **Value**: The actual book content — "The information I'll share if you attend to me"

Why separate projections instead of just using the raw embeddings?  
Because what a token IS (its embedding) is different from:  
- What it's LOOKING FOR (query)
- What it OFFERS to others (key)
- What information it PASSES ALONG when attended to (value)

The linear projections learn these roles during training.

## Why no bias in the linear projections?

```python
nn.Linear(n_embd, head_size, bias=False)
```

Standard practice in transformer attention layers. The bias would add a constant to every attention score regardless of content — it doesn't help and adds parameters. Most modern implementations omit it.

## Why register_buffer for the mask?

```python
self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
```

`register_buffer` vs `nn.Parameter`:  
- `nn.Parameter`: stored in model, included in `model.parameters()`, updated by optimizer  
- `register_buffer`: stored in model, NOT in `model.parameters()`, NOT updated by optimizer  

The causal mask is a fixed constant — it should never be learned or changed. But we want it to:
1. Move to the correct device with `model.to(device)`
2. Be saved and loaded with `torch.save` / `torch.load`

`register_buffer` gives us both without making it a trainable parameter.

## Why torch.tril?

```python
torch.tril(torch.ones(block_size, block_size))
```

`tril` = lower triangular. For block_size=4:
```
[[1, 0, 0, 0],
 [1, 1, 0, 0],
 [1, 1, 1, 0],
 [1, 1, 1, 1]]
```

Row i, column j = 1 if j <= i (can attend to past and present), 0 if j > i (cannot attend to future).  
Token 0 can only see itself. Token 3 can see tokens 0,1,2,3.

## Why divide by √head_size?

```python
wei = q @ k.transpose(-2, -1) * head_size**-0.5
```

**The problem without scaling:**  
If Q and K have unit variance, their dot product has variance = head_size.  
For head_size=64, dot products have std=8 — they can be very large (e.g. 20, -15, 18, -12).  
When you apply softmax to such extreme values, the output becomes almost a one-hot vector:  
`softmax([20, -15, 18]) ≈ [0.88, 0.0, 0.12]`  
This means the model only attends to one token, losing the ability to spread attention.  
Gradients also vanish because softmax is saturated.

**The fix:**  
Dividing by √head_size brings the variance back to 1, keeping softmax in a "smooth" regime.

Proof: if Q,K ~ N(0,1), then Q·K ~ N(0, head_size). Dividing by √head_size gives N(0,1).

## Why masked_fill with -inf?

```python
wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
```

We want future positions to have zero attention weight after softmax.  
`softmax(-inf) = exp(-inf) / sum = 0 / sum = 0` ✓  

Why not just set to 0 before softmax?  
Because softmax normalises the entire row. Setting a logit to 0 before softmax would give it `exp(0) = 1` — that's not zero attention!  
Setting to -inf gives `exp(-inf) = 0` — exactly what we want.

## Why [:T, :T] instead of the full mask?

```python
self.tril[:T, :T]
```

The mask is pre-allocated at `block_size × block_size` (e.g. 256×256).  
But the current sequence might be shorter than block_size (e.g. T=10 during generation).  
We slice `[:T, :T]` to get only the relevant portion.

## Why dropout on attention weights?

```python
wei = self.dropout(wei)
```

This randomly zeroes some attention connections during training.  
Prevents the model from over-relying on specific token-to-token relationships.  
Forces the model to learn redundant attention patterns — more robust representations.  
At test time (model.eval()), dropout is disabled — all connections are used.

---
# PART 5 — MULTI-HEAD ATTENTION
---

## Why multiple heads?

A single attention head learns ONE type of relationship between tokens.  
Think about what relationships exist in language:
- Syntactic: subject attends to its verb
- Semantic: pronoun attends to the noun it refers to
- Positional: each token attends to the previous token
- Structural: closing bracket attends to opening bracket

These are all different relationships that happen simultaneously.  
Multiple heads let the model attend to different aspects of the context in parallel.

In practice, different heads DO specialise. Research (Voita et al. 2019) found that specific heads in GPT-2 learn specific functions: positional heads, syntactic heads, rare-word heads.

## Why head_size = n_embd // n_heads?

```python
head_size = n_embd // n_heads
```

We want the total computation of multi-head attention to equal the computation of one full attention head.  
If n_embd=384 and n_heads=6: each head works in 384/6=64 dimensions.  
Concatenating 6 heads of 64 dims = 384 dims total — same as one head of 384 dims.

This makes multi-head attention the same cost as single-head attention but with richer representations.

## Why a projection after concatenation?

```python
self.proj = nn.Linear(n_heads * head_size, n_embd)
```

After concatenating all head outputs, we have a vector of size `n_heads × head_size = n_embd`.  
Each head computed its output independently — they don't know about each other.  
The projection layer mixes information across all heads, letting the model combine insights from different heads.

This is also the "output projection" W_O from the original Attention is All You Need paper.

## Why nn.ModuleList instead of a Python list?

```python
self.heads = nn.ModuleList([Head(...) for _ in range(n_heads)])
```

If you use a regular Python list `[Head(...)]`, PyTorch doesn't know these are submodules.  
Consequences:
- `model.parameters()` won't include their parameters → optimizer won't update them
- `model.to(device)` won't move them to GPU
- `model.state_dict()` won't save them

`nn.ModuleList` registers them properly as PyTorch submodules, so all of the above work correctly.

---
# PART 6 — FEED-FORWARD NETWORK
---

## Why have an FFN at all? Isn't attention enough?

Attention and FFN do fundamentally different things:

**Attention** = **communication** — tokens exchange information with each other.  
After attention, token i has a new representation that incorporates information from tokens j, k, l...  
But this new representation is a weighted average of other tokens' values — it's a mixing operation.

**FFN** = **computation / transformation** — each token processes its own representation independently.  
After attention has gathered the relevant information, the FFN transforms it.  
Think of attention as "read from memory" and FFN as "process what you just read."

Research shows that FFN layers store factual knowledge (Geva et al. 2021 — "Transformer Feed-Forward Layers Are Key-Value Memories"). When GPT-3 knows that "Paris is the capital of France," that knowledge is stored in FFN weights.

## Why 4× expansion?

```python
nn.Linear(n_embd, 4 * n_embd)
```

This comes directly from the original Transformer paper (Vaswani et al. 2017). They used d_model=512 and inner dimension=2048 — exactly 4×. It has become a standard convention.

Why 4×? It gives the FFN enough capacity to store and retrieve knowledge without being too expensive.  
Experiments with 2×, 4×, 8× show diminishing returns beyond 4×.

## Why ReLU and not other activations?

```python
nn.ReLU()
```

ReLU(x) = max(0, x) — simple, fast, doesn't saturate for positive values.  

GPT-2 actually uses **GELU** (Gaussian Error Linear Unit), which is smoother than ReLU.  
We use ReLU here for simplicity — it's slightly worse but much easier to understand.  

If you want to match GPT-2 exactly, replace `nn.ReLU()` with `nn.GELU()` — one character change.

---
# PART 7 — TRANSFORMER BLOCK
---

## Why residual connections?

```python
x = x + self.sa(self.ln1(x))   # residual
x = x + self.ffn(self.ln2(x))  # residual
```

Without residuals (plain deep network):
```
x_out = FFN_6(FFN_5(FFN_4(FFN_3(FFN_2(FFN_1(x_in))))))
```
Gradient from loss must pass through all 6 layers to reach FFN_1.  
Each layer can shrink or distort the gradient → **vanishing gradients** in early layers.  
Early layers barely learn → deep networks fail.

With residuals:
```
x = x + FFN_1(x)  # x is passed directly to the next layer
x = x + FFN_2(x)  # again
...
```
The gradient can flow directly from the loss to ANY layer via the `+x` path.  
It doesn't have to pass through the FFN — it shortcuts around it.  
This is why ResNets (He et al. 2016) could train 100+ layer networks when plain networks failed at 20 layers.

**Another way to think about it:** Each block learns a small CORRECTION to x, not a complete transformation. `x_new = x + small_delta(x)`. This is much easier to learn than learning the full transformation from scratch.

## Why LayerNorm? Why not BatchNorm?

```python
self.ln1 = nn.LayerNorm(n_embd)
```

**BatchNorm** normalises across the BATCH dimension — uses statistics from other examples in the batch.  
Problems for language models:
1. Different sequence lengths make batch statistics noisy
2. At inference time with batch_size=1, there's no batch to compute statistics from
3. Behaviour differs between train (batch stats) and eval (running stats) — unstable

**LayerNorm** normalises across the FEATURE dimension — uses statistics from the current token only.
```
LayerNorm(x) = (x - mean(x)) / std(x) * γ + β
```
Where γ (scale) and β (shift) are learnable parameters.

Benefits:
- Works with any batch size including 1
- Same behaviour at train and eval time
- Stable regardless of sequence length

## Why pre-norm instead of post-norm?

**Original Transformer (post-norm):**
```python
x = LayerNorm(x + Attention(x))   # norm AFTER residual
```

**GPT-2 (pre-norm):**
```python
x = x + Attention(LayerNorm(x))   # norm BEFORE attention
```

Pre-norm is more stable because:
- The residual stream `x` is never normalised — it can grow throughout training
- This prevents gradient vanishing in very deep networks
- Training is more stable and requires less careful LR tuning

Post-norm gives slightly better final performance IF training is stable, but pre-norm is much easier to train. Modern LLMs (GPT-2 onwards) all use pre-norm.

---
# PART 8 — FULL GPT MODEL
---

## Why both token AND position embeddings?

```python
tok_emb = self.transformer.wte(idx)   # token embedding
pos_emb = self.transformer.wpe(pos)   # position embedding
x = tok_emb + pos_emb
```

**Token embedding** answers: "What is this token semantically?"  
Token 'A' always maps to the same vector regardless of where it appears.

**Position embedding** answers: "Where in the sequence is this token?"  
Position 0 always maps to the same vector regardless of what token is there.

**Why do we need both?**  
Self-attention is permutation-invariant: `Attention({A,B,C}) = Attention({C,A,B})`.  
Without position information, 'dog bites man' and 'man bites dog' would produce identical representations.  
Adding position embeddings breaks this symmetry.

**Why ADD instead of concatenate?**  
Concatenating would double the dimension: `[tok_emb, pos_emb]` would be 2×n_embd.  
Addition keeps the dimension the same: n_embd.  
Both approaches work — GPT uses addition for simplicity and efficiency.

## Why learnable position embeddings instead of sinusoidal?

The original Transformer used fixed sinusoidal position embeddings:
```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

GPT uses learned position embeddings (`nn.Embedding(block_size, n_embd)`).  
The original paper found both work equally well.  
Learned embeddings are simpler to implement and can potentially learn task-specific positional patterns.  
Downside: fixed max context length (block_size). Sinusoidal can extrapolate beyond training length.

## Why weight tying?

```python
self.transformer.wte.weight = self.lm_head.weight
```

The token embedding table (wte) maps token_id → vector.  
The LM head maps vector → token_id probabilities.  
These are inverse operations — it makes sense they should use related weights.

**Intuition:** If the embedding of 'king' is close to 'queen', then the LM head should also predict 'queen' as likely when the hidden state is similar to the 'king' embedding.

**Practical benefit:** Saves `vocab_size × n_embd` parameters.  
For our model: 65 × 384 = 24,960 parameters saved — not huge, but meaningful for large vocab models.

Press & Wolf (2017) showed weight tying slightly IMPROVES performance — the shared constraint acts as regularisation.

## Why _init_weights?

```python
nn.init.normal_(module.weight, mean=0.0, std=0.02)
```

PyTorch's default initialisation is fine for simple networks but can cause instability in deep transformers.

std=0.02 comes from GPT-2. Why 0.02?  
With 6 layers, each residual addition accumulates variance. If each layer adds variance σ², after 6 layers we have variance 6σ².  
Setting σ=0.02 keeps the residual stream variance reasonable at ~6×0.0004=0.0024.

**Zero bias init:** Linear biases start at 0 — they'll be learned from there. Starting at non-zero could create systematic biases that are hard to unlearn.

## Why assert T <= block_size?

```python
assert T <= self.config.block_size
```

The position embedding table has exactly `block_size` rows.  
If T > block_size, `wpe(pos)` would request index `block_size` which doesn't exist → silent index error.  
The assert catches this with a clear error message instead of a confusing crash.

Also: the causal mask is pre-allocated at `block_size × block_size`.  
Sequences longer than block_size would try to use rows/cols that don't exist in the mask.

## Why crop context in generate()?

```python
idx_cond = idx[:, -self.config.block_size:]
```

During generation, the sequence grows by 1 token at each step.  
After `block_size` steps, the sequence is longer than the model's context window.  
We crop to the last `block_size` tokens — the model can only "see" this much anyway.  
Without this crop, `wpe` would be called with position index > block_size → error.

## Why temperature in generation?

```python
logits = logits[:, -1, :] / temperature
```

Dividing logits by temperature before softmax:

- **temperature < 1** (e.g. 0.5): logits become MORE extreme → softmax more peaked → model more confident → less diverse but more coherent
- **temperature = 1.0**: no change — original distribution
- **temperature > 1** (e.g. 1.5): logits become LESS extreme → softmax more uniform → more random → more creative but potentially incoherent

Example: logits = [3, 1, -1]
```
temp=0.5: softmax([6, 2, -2])  = [0.98, 0.02, 0.00]  (almost deterministic)
temp=1.0: softmax([3, 1, -1])  = [0.84, 0.11, 0.02]  (confident)
temp=2.0: softmax([1.5, 0.5, -0.5]) = [0.61, 0.27, 0.10]  (uncertain)
```

## Why top_k?

```python
if top_k is not None:
    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
    logits[logits < v[:, [-1]]] = float('-inf')
```

Even with low temperature, the model might occasionally sample from very unlikely tokens (probability 0.001) — producing gibberish.  
top_k=40 means: only consider the 40 most likely tokens. Zero out the rest.  
This prevents the model from ever generating from the long tail of unlikely tokens.  
GPT-2 uses top_k=40 by default. Common values: 20–100.

---
# PART 9 — TRAINING LOOP
---

## Why AdamW and not SGD or Adam?

```python
optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr,
                               betas=(0.9, 0.95), weight_decay=0.1)
```

**SGD**: Simple but requires careful LR tuning. Doesn't adapt per-parameter. Slow to converge on transformers.

**Adam**: Maintains per-parameter adaptive learning rates using first and second moment estimates. Much faster convergence than SGD for transformers.

**AdamW**: Adam + fixed weight decay.  
Regular Adam incorporates weight decay into the gradient update in a mathematically incorrect way (Loshchilov & Hutter 2017 showed this).  
AdamW applies weight decay directly to weights, separately from the gradient update — mathematically correct L2 regularisation.

## Why betas=(0.9, 0.95)?

Adam's betas control exponential moving averages:
- **β1=0.9**: Smoothing factor for gradient (first moment). 90% old estimate + 10% new gradient.
- **β2=0.95**: Smoothing factor for gradient² (second moment). 95% old + 5% new.

PyTorch default is β2=0.999 (used in original Adam paper for image classification).  
GPT-2 uses β2=0.95 — a lower value makes the second moment respond faster to gradient changes, which helps for language modelling where gradient magnitudes vary more.

## Why weight_decay=0.1?

Weight decay adds a penalty: `loss_total = loss_task + λ × ||weights||²`  
This pushes weights toward zero, preventing any single weight from becoming too large.  
It's a regulariser that reduces overfitting.

**Important detail:** We should apply weight decay to weights but NOT to:
- Biases (1D parameters)
- LayerNorm scale/shift parameters (γ, β)
- Embeddings

For simplicity, we apply it to all parameters here. In production GPT-2 code, there's a parameter group split.

## Why cosine LR schedule?

**Constant LR**: Simple but suboptimal. Optimal LR at the start (fast progress) is too large near convergence (overshooting).

**Step decay**: LR halved every N steps. Requires manual tuning of N.

**Cosine decay**: Smooth decrease following cosine curve. Widely used in LLM training.
```
lr = min_lr + 0.5 × (max_lr - min_lr) × (1 + cos(π × progress))
```
At progress=0: lr = min_lr + 0.5 × (max_lr-min_lr) × 2 = max_lr ✓  
At progress=1: lr = min_lr + 0.5 × (max_lr-min_lr) × 0 = min_lr ✓

## Why linear warmup?

At the start of training, model weights are random. Gradients can be very large and noisy.  
Starting with a large LR causes large weight updates from noisy gradients → divergence.  

Warmup: start with very small LR (near 0), linearly increase to max_lr over warmup_steps.  
By the time we reach max_lr, the model has stabilised and gradients are more meaningful.

## Why min_lr = max_lr / 10?

```python
max_lr=3e-4, min_lr=3e-5
```

Conventional rule: min_lr = max_lr / 10.  
We don't want the LR to reach exactly 0 at the end — some learning signal is still useful.  
The 10× ratio is from Chinchilla scaling laws (Hoffmann et al. 2022).

## Why gradient clipping?

```python
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
```

In deep networks, gradients can occasionally become very large ("exploding gradients").  
This happens when the loss surface has very steep regions.  
A single large gradient update can throw the model into a bad region it can't recover from.

`clip_grad_norm_(params, max_norm=1.0)` computes the global gradient norm and, if it exceeds 1.0, scales all gradients down proportionally.  

This doesn't change the DIRECTION of the update, only its magnitude.  
Threshold of 1.0 is standard for language models. GPT-3 uses 1.0.

## Why set_to_none=True in zero_grad?

```python
optimizer.zero_grad(set_to_none=True)
```

`set_to_none=False` (default): sets gradient tensors to zero tensors.  
`set_to_none=True`: frees the gradient tensors entirely (sets to None).  

Setting to None frees that GPU memory immediately. On a GPU with limited VRAM (like Colab T4), this saves ~10-20% memory — enough to fit a larger batch or model.

## Why max_lr=3e-4 specifically?

3e-4 (0.0003) is empirically one of the best learning rates for Adam/AdamW on transformer models.  
Karpathy calls it "the best LR" and it appears across many papers.  
It's large enough for fast learning but small enough to be stable.

For very large models (GPT-3 scale), LR is scaled down (GPT-3 used 6e-5).

---
# PART 10 — HYPERPARAMETERS
---

## n_embd = 384 — Why this number?

n_embd is the size of every token representation throughout the model.  
384 = 6 × 64. We chose it because:
1. Divisible by n_heads=6 (required: head_size = n_embd // n_heads must be integer)
2. Large enough to represent rich token relationships on Tiny Shakespeare
3. Small enough to train quickly on Colab T4

GPT-2 small uses n_embd=768. GPT-3 uses 12,288. Larger = more capacity but slower.

## n_layers = 6 — Why 6 blocks?

Each additional layer lets the model compute more complex functions.  
Layer 1 might learn character n-grams.  
Layer 2 might learn word patterns.  
Layer 3 might learn phrase structures.  
...and so on.

6 is the sweet spot for Tiny Shakespeare — enough depth to generate coherent text,  
small enough to train in ~15 minutes on Colab.

GPT-2 small: 12 layers. GPT-3: 96 layers.

## n_heads = 6 — Why 6 attention heads?

Must divide n_embd evenly: 384 / 6 = 64 (head_size=64). ✓  
6 heads = 6 different relationship types the model can learn simultaneously.  
Research shows diminishing returns beyond ~8 heads for models this size.

GPT-2 small: 12 heads. We use 6 because n_embd=384 < 768.

## block_size = 256 — Why 256 context tokens?

This is the maximum number of tokens the model considers when making each prediction.  
256 characters ≈ 2-3 lines of Shakespeare — enough context for coherent text generation.  
Larger context = more memory (attention is O(T²)) and slower training.

Memory for attention: `B × n_heads × T × T × 4 bytes`  
At T=256: 64 × 6 × 256 × 256 × 4 = ~100 MB — manageable on T4.  
At T=1024: 64 × 6 × 1024 × 1024 × 4 = ~1.6 GB — would OOM.

GPT-2: 1024 tokens. GPT-3: 2048. GPT-4: ~8192+.

## dropout = 0.2 — Why 20%?

Dropout rate = probability of zeroing each activation.  
0.2 means 20% of neurons are randomly disabled during each forward pass.

Too high (>0.5): model can't learn — too much information is dropped.  
Too low (<0.1): not enough regularisation — overfitting on small datasets.  
0.2 is standard for small transformer models on small datasets.  

GPT-2 uses 0.1 (trained on much more data, needs less regularisation).

## batch_size = 64 — Why 64?

Larger batch = more stable gradient estimate = smoother training.  
Larger batch = more GPU memory used per step.  
64 sequences × 256 tokens × 384 floats × 4 bytes ≈ 25 MB activations — fits on T4.

Rule of thumb: use the largest batch size that fits in GPU memory.

## max_iters = 5000 — Why 5000 steps?

5000 steps × 64 batch × 256 tokens = 81.9M token-predictions trained on.  
Tiny Shakespeare has ~900K training tokens.  
So we train on each token ~90 times on average (90 epochs approximately).

This is enough to get good results without overfitting too badly.  
Signs of overfitting: train loss keeps dropping but val loss stops improving or rises.

## eval_interval = 500 — Why evaluate every 500 steps?

Evaluating too frequently wastes compute (estimate_loss runs 200 batches).  
Evaluating too infrequently means we can't monitor training.  
500 steps gives 10 evaluation points over 5000 steps — enough to see the learning curve shape.

---
# PART 11 — ABLATION STUDY REASONING
---

## Why run ablations?

Anyone can run Karpathy's NanoGPT notebook and get a trained model.  
What distinguishes a researcher/engineer from a tutorial follower is asking:  
**"What if I change X? Why does Y happen?"**

Ablation studies are standard in ML papers. Every major paper has a Table like:  
"Without component X, performance drops by Y" — proving X is necessary.

## Why vary n_layers first?

Depth is the most impactful hyperparameter in transformers.  
Each layer adds representational capacity.  
Expected finding: more layers → lower perplexity, BUT with diminishing returns.  
We test 2, 4, 6 layers to show this curve.

## Why vary n_heads?

There's a common misconception that more heads = better.  
In practice, beyond a certain point adding heads doesn't help because the total attention capacity (n_embd) is fixed — you're just dividing it into smaller subspaces.  
Expected finding: 4-6 heads optimal for this model size.

## Why vary block_size (context window)?

Longer context = model can use more history = should improve perplexity.  
But: longer context = O(T²) attention = slower training = fewer steps in same time.  
This ablation reveals the trade-off between context length and training efficiency.

Expected finding: block_size=256 significantly better than 64, showing long-range dependencies matter in Shakespeare.

## Why keep other parameters fixed during each ablation?

The cardinal rule of controlled experiments: **vary one thing at a time**.  
If you change n_layers AND n_heads simultaneously and performance improves, you don't know which caused it.  
By fixing everything except the variable of interest, changes in perplexity are directly attributable to that variable.

## Why use the same random seed for all ablation runs?

```python
torch.manual_seed(1337)
```

Neural network training has randomness: weight initialisation, batch ordering, dropout.  
If we don't fix the seed, two identical models might get different perplexities just due to random chance.  
Fixing the seed ensures differences between runs are due to architectural changes, not luck.

---
# PART 12 — PYTORCH-SPECIFIC REASONING
---

## What is nn.ModuleDict?

```python
self.transformer = nn.ModuleDict(dict(
    wte  = nn.Embedding(...),
    wpe  = nn.Embedding(...),
    drop = nn.Dropout(...),
    h    = nn.ModuleList([...]),
    ln_f = nn.LayerNorm(...),
))
```

`nn.ModuleDict` is a dict that properly registers its values as PyTorch submodules.  
Like `nn.ModuleList` but with string keys instead of integer indices.  
You access components with `self.transformer.wte` — clean, readable, and PyTorch-aware.

## What does .view() do?

```python
logits.view(B*T, C)
```

`.view()` reshapes a tensor WITHOUT copying data.  
The total number of elements must stay the same: B×T×C = B*T × C ✓  
Think of it as changing how you index into the same block of memory.

`.reshape()` is similar but copies data if necessary (when tensor is not contiguous).  
`.view()` is faster but requires the tensor to be contiguous in memory.  
For our case, the tensor is always contiguous so `.view()` works fine.

## What does .transpose(-2, -1) do?

```python
k.transpose(-2, -1)
```

For k of shape `(B, T, head_size)`:  
`.transpose(-2, -1)` swaps the last two dimensions → `(B, head_size, T)`

Then `q @ k.transpose(-2, -1)`:  
`(B, T, hs) @ (B, hs, T)` → `(B, T, T)`

The `@` operator does batched matrix multiplication — the B dimension is preserved,  
and matrix multiplication is done on the last two dimensions.

## What is torch.arange?

```python
pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
```

Creates a 1D tensor `[0, 1, 2, ..., T-1]`.  
These are the position indices passed to the position embedding.  
`device=idx.device` ensures the positions tensor is on the same device (GPU/CPU) as the input — you can't do operations between tensors on different devices.

## What is .unsqueeze(0) in generate()?

```python
feat_t = torch.tensor(feats).unsqueeze(0)
```

Our model expects input of shape `(B, T)` — batch dimension first.  
During generation we have one sequence of shape `(T,)`.  
`.unsqueeze(0)` adds a batch dimension at position 0: `(T,)` → `(1, T)`  
Now B=1 — a batch of size 1.

## What is .tolist()?

```python
decode(model.generate(context, max_new_tokens=300)[0].tolist())
```

`model.generate()` returns a tensor on the GPU.  
`[0]` takes the first (and only) batch item — shape goes from `(1, T)` to `(T,)`  
`.tolist()` converts the tensor to a Python list — required because `decode()` expects a list, not a tensor.  
This also implicitly moves the data from GPU to CPU.

---
# PART 13 — METRICS REASONING
---

## Why cross-entropy loss?

Language modelling is a classification problem: at each position, classify which of the `vocab_size` tokens comes next.  
Cross-entropy is the standard loss for multi-class classification.  

Information-theoretic interpretation: cross-entropy measures how many bits you need to encode the true next token given the model's probability distribution. Lower = more efficient = better model.

## What is perplexity and why report it?

```python
perplexity = math.exp(cross_entropy_loss)
```

**Perplexity** is the exponentiated cross-entropy. Intuition:  
"Perplexity of N means the model is as confused as if choosing uniformly between N equally likely options."

| Model | Loss | Perplexity | Interpretation |
|-------|------|-----------|----------------|
| Random | 4.17 | 65 | Guessing uniformly among 65 chars |
| Bigram | ~2.5 | ~12 | Like choosing among 12 chars |
| NanoGPT | ~1.5 | ~4.5 | Like choosing among 4-5 chars |
| Perfect | 0 | 1 | Always predicts the exact next char |

Why report perplexity instead of loss?  
Perplexity is more **interpretable** — you can intuitively understand "choosing among 4.5 characters".  
Loss (1.5) has no intuitive meaning by itself.  
Perplexity is also a **standard benchmark** — papers compare on perplexity so you can look up how your model compares.

## Why report both train and val loss?

- **Train loss decreasing**: model is learning ✓
- **Val loss also decreasing**: model is generalising ✓
- **Val loss stops decreasing while train continues**: overfitting — model memorising training data ✗
- **Val loss << train loss**: something wrong — data leakage or bug
- **Both losses high**: underfitting — model/training has a problem

The gap between train and val loss is the **generalisation gap** — smaller is better.

---
# PART 14 — INTERVIEW CHEAT SHEET
---

These are the questions you will be asked in interviews. Now you have real answers from implementing it yourself.

---

**Q: What is the attention mechanism?**  
A: Each token computes Q, K, V vectors via learned linear projections. Attention weights are computed as scaled dot products of Q and K, masked for causality, then softmaxed. The output is a weighted sum of V vectors. This lets every token directly attend to any past token.

**Q: Why scale by 1/√d_k?**  
A: Q·K dot products have variance = d_k. Without scaling, large d_k causes extreme logits before softmax, making it nearly one-hot and vanishing gradients. Dividing by √d_k normalises variance back to 1.

**Q: What's the difference between encoder and decoder transformers?**  
A: Encoder (BERT): bidirectional attention — every token sees every other. Used for understanding tasks. Decoder (GPT): causal/masked attention — each token only sees past tokens. Used for generation.

**Q: Why residual connections?**  
A: Allow gradients to flow directly from loss to early layers without passing through all intermediate layers. Prevents vanishing gradients in deep networks. Each block learns small corrections to x, not full transformations.

**Q: Why LayerNorm and not BatchNorm?**  
A: LayerNorm normalises within a single example across features. BatchNorm normalises across the batch. For variable-length sequences and batch_size=1 inference, LayerNorm is stable; BatchNorm is not.

**Q: What does temperature do in generation?**  
A: Divides logits before softmax. Low temp (<1): sharper distribution, more repetitive. High temp (>1): flatter distribution, more creative but potentially incoherent.

**Q: What is weight tying?**  
A: Sharing weights between the input token embedding and the output LM head. Saves vocab_size×n_embd parameters. Slightly improves performance as it constrains the model — the embedding that maps token→vector and the head that maps vector→token should be related.

**Q: What is the FFN for?**  
A: Attention mixes information across tokens (communication). FFN transforms each token's representation independently (computation). Research shows FFN layers store factual knowledge.

**Q: Why multi-head attention?**  
A: A single head learns one relationship type. Multiple heads can specialise — some learn syntactic relationships, some semantic, some positional. Total compute is the same as one full-size head.

**Q: What is perplexity?**  
A: exp(cross_entropy_loss). Interpretable measure: perplexity N means the model is as confused as choosing uniformly among N options. Lower is better.

**Q: What did your ablation study find?**  
A: [Fill in after running — e.g. "Increasing context window from 64 to 256 gave the largest perplexity improvement (X→Y), showing Shakespeare has long-range dependencies. Adding layers beyond 4 gave diminishing returns."]

**Q: How does gradient clipping work?**  
A: Computes the global L2 norm of all gradients. If it exceeds the threshold (1.0), scales all gradients down proportionally. Prevents a single large gradient from causing an explosive weight update.

**Q: What is the difference between Adam and AdamW?**  
A: Adam incorrectly incorporates weight decay into the adaptive learning rate. AdamW applies weight decay separately as direct parameter shrinkage — mathematically correct L2 regularisation.

**Q: Why cosine learning rate schedule?**  
A: Starts high for fast initial learning, gradually decreases for fine-tuning near convergence. The smooth cosine curve avoids the abrupt drops of step decay. Linear warmup at the start prevents large updates when gradients are noisy from random initialisation.